# Visualize results using forward matrix

- Load forward matrix and parcellations
- Load NIRS toolbox results
- Directly project results to cortical surface using adot matrix


In [1]:
import glob
import os
import re 
import json 

import cedalion
import cedalion.dataclasses
import cedalion.dot
import cedalion.io
import cedalion.io.snirf
import cedalion.vis.anatomy
import cedalion.vis.blocks as vbx
from cedalion import units

import pyvista as pv
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xarray as xr

np.set_printoptions(suppress=True)

# BIDS paths
task      = "complexwalk"
session   = "protocol1"
bids_root = "../../Park-MOVE_fnirs_dataset_v2/fNIRS_data/bids_dataset_snirf"
fwm_dir   = "../data/forward_model"
os.makedirs(fwm_dir, exist_ok=True)

# What counts as a short channel?
DIST_THRESHOLD = 1.5 * units.cm 

# Get all SNIRF files
snirf_files = sorted(glob.glob(
    f"{bids_root}/sub-*/ses-{session}/nirs/sub-*_task-{task}_nirs.snirf"
))
print(f"Found {len(snirf_files)} subjects")

Found 130 subjects


# Load montage and forward matrix data

In [2]:
# Load amp data
fname = snirf_files[0]
rec = cedalion.io.snirf.read_snirf(fname)[0]
amp = rec.get_timeseries()

# Load forward model
sensitivity_file = f"{fwm_dir}/ParkMOVE_colin27_Adot.h5"
Adot = cedalion.io.load_Adot(sensitivity_file)
print("Adot dims:", dict(Adot.sizes))

# Get long channels
amp_long, amp_short = cedalion.nirs.split_long_short_channels(
    amp, rec.geo3d, DIST_THRESHOLD
)

# Extra filtering for D>7
detector_ids = np.array([
    int(re.search(r"D(\d+)$", ch).group(1))
    for ch in amp.channel.values
])
extra_short = amp.channel.values[detector_ids >= 8]

# Union of both definitions
all_short = np.union1d(
    amp_short.channel.values,
    extra_short
)

all_long = np.setdiff1d(
    amp.channel.values,
    all_short
)

amp_short = amp.sel(channel=all_short)
amp_long = amp.sel(channel=all_long)

print(f"Long channels: {amp_long.sizes['channel']}")
print(f"Short channels: {amp_short.sizes['channel']}")

# Restrict Adot to long channels
shared_channels = np.intersect1d(Adot.channel.values, amp_long.channel.values)
Adot_long = Adot.sel(channel=shared_channels)
print(f"Long channels in Adot: {len(shared_channels)}")

Adot dims: {'channel': 33, 'vertex': 35050, 'wavelength': 2}
Long channels: 20
Short channels: 13
Long channels in Adot: 20


# Prepare parcellation and function for getting boundaries

In [3]:
colin_ijk = cedalion.dot.get_standard_headmodel("colin27")
brodmann_voxel_label_niftii, brodmann_labels_json = cedalion.data.get_atlas_files("brodmann")

# dictionary to map numeric voxel labels in nifti to string labels
with brodmann_labels_json.open("r") as fin:
    brodmann_num2label = json.load(fin)
    brodmann_num2label = {i["index"] : i["name"] for i in brodmann_num2label["labels"]}

# show some entries of the dictionary
print(brodmann_num2label)

colin_ijk_labeled = colin_ijk.assign_parcels_via_mni_coords(
    coordinate_label="parcel_brodmann",
    label_mapping=brodmann_num2label,
    voxel_label_niftii=brodmann_voxel_label_niftii,
    voxel_label_crs="mni152",
    mni_eps=5
)

def get_parcel_boundary_edges(brain_surface, label_coord, labels=None):

    edges = brain_surface.mesh.edges_unique
    vertex_labels = brain_surface.vertices.coords[label_coord].values

    lab0 = vertex_labels[edges[:, 0]]
    lab1 = vertex_labels[edges[:, 1]]
    is_boundary = lab0 != lab1

    if labels is not None:
        labels = set(labels)
        touches_target = np.isin(lab0, list(labels)) | np.isin(lab1, list(labels))
        is_boundary &= touches_target

    return edges[is_boundary]

{1: 'right_Ent', 2: 'right_BA20', 3: 'right_BA38', 4: 'right_BA21', 5: 'right_BA36', 6: 'right_BA11', 7: 'right_BA37', 8: 'right_BA25', 9: 'right_BA12', 10: 'right_BA19', 11: 'right_BA47', 12: 'right_BA22', 13: 'right_BA18', 14: 'right_BA10', 15: 'right_BA17', 16: 'right_BA32', 17: 'right_BA24', 18: 'right_BA46', 19: 'right_BA45', 20: 'right_BA33', 21: 'right_BA44', 22: 'right_BA6', 23: 'right_BA23', 24: 'right_BA42', 25: 'right_BA43', 26: 'right_BA30', 27: 'right_BA41', 28: 'right_BA29', 29: 'right_BA26', 30: 'right_BA31', 31: 'right_BA4', 32: 'right_BA1', 33: 'right_BA9', 34: 'right_BA39', 35: 'right_BA2', 36: 'right_BA3', 37: 'right_BA40', 38: 'right_BA7', 39: 'right_BA8', 40: 'right_BA5', 41: 'right_BA52', 42: 'right_BA35', 43: 'right_BA34', 101: 'left_Ent', 102: 'left_BA20', 103: 'left_BA38', 104: 'left_BA21', 105: 'left_BA36', 106: 'left_BA11', 107: 'left_BA37', 108: 'left_BA25', 109: 'left_BA12', 110: 'left_BA19', 111: 'left_BA47', 112: 'left_BA22', 113: 'left_BA18', 114: 'left_

# Load t-stats from NIRS toolbox - condition values

In [4]:
# Load all condition results
groupstats_protocol_2_oa = pd.read_csv("../data/groupstats_oa_protocol_2.csv")
groupstats_protocol_2_pd = pd.read_csv("../data/groupstats_pd_protocol_2.csv")
groupstats_protocol_3_oa = pd.read_csv("../data/groupstats_oa_protocol_3.csv")
groupstats_protocol_3_pd = pd.read_csv("../data/groupstats_pd_protocol_3.csv")

protocol_2_oa_st_walk_cluster_1 = groupstats_protocol_2_oa[
    groupstats_protocol_2_oa["cond"] == "Straight_walking:cluster_1"
].copy()
protocol_2_oa_st_walk_cluster_2 = groupstats_protocol_2_oa[
    groupstats_protocol_2_oa["cond"] == "Straight_walking:cluster_2"
].copy()
protocol_2_oa_st_nav_cluster_1 = groupstats_protocol_2_oa[
    groupstats_protocol_2_pd["cond"] == "Navigated_walking:cluster_1"
].copy()
protocol_2_oa_st_nav_cluster_2 = groupstats_protocol_2_oa[
    groupstats_protocol_2_pd["cond"] == "Navigated_walking:cluster_2"
].copy()

protocol_2_pd_st_walk_cluster_1 = groupstats_protocol_2_pd[
    groupstats_protocol_2_pd["cond"] == "Straight_walking:cluster_1"
].copy()
protocol_2_pd_st_walk_cluster_2 = groupstats_protocol_2_pd[
    groupstats_protocol_2_pd["cond"] == "Straight_walking:cluster_2"
].copy()
protocol_2_pd_st_nav_cluster_1 = groupstats_protocol_2_pd[
    groupstats_protocol_2_pd["cond"] == "Navigated_walking:cluster_1"
].copy()
protocol_2_pd_st_nav_cluster_2 = groupstats_protocol_2_pd[
    groupstats_protocol_2_pd["cond"] == "Navigated_walking:cluster_2"
].copy()

protocol_3_oa_st_nav_cluster_1 = groupstats_protocol_3_oa[
    groupstats_protocol_3_oa["cond"] == "Navigation:cluster_1"
].copy()
protocol_3_oa_st_nav_cluster_2 = groupstats_protocol_3_oa[
    groupstats_protocol_3_oa["cond"] == "Navigation:cluster_2"
].copy()
protocol_3_oa_dt_nav_cluster_1 = groupstats_protocol_3_oa[
    groupstats_protocol_3_oa["cond"] == "Navigation_and_Aud_Stroop:cluster_1"
].copy()
protocol_3_oa_dt_nav_cluster_2 = groupstats_protocol_3_oa[
    groupstats_protocol_3_oa["cond"] == "Navigation_and_Aud_Stroop:cluster_2"
].copy()

protocol_3_pd_st_nav_cluster_1 = groupstats_protocol_3_pd[
    groupstats_protocol_3_pd["cond"] == "Navigation:cluster_1"
].copy()
protocol_3_pd_st_nav_cluster_2 = groupstats_protocol_3_pd[
    groupstats_protocol_3_pd["cond"] == "Navigation:cluster_2"
].copy()
protocol_3_pd_dt_nav_cluster_1 = groupstats_protocol_3_pd[
    groupstats_protocol_3_pd["cond"] == "Navigation_and_Aud_Stroop:cluster_1"
].copy()
protocol_3_pd_dt_nav_cluster_2 = groupstats_protocol_3_pd[
    groupstats_protocol_3_pd["cond"] == "Navigation_and_Aud_Stroop:cluster_2"
].copy()


# Load t-stats from NIRS toolbox - contrasts

In [5]:
# Load all contrast results
groupstats_protocol_2_oa = pd.read_csv("../data/groupstats_oa_protocol_2_contrast.csv")
groupstats_protocol_2_pd = pd.read_csv("../data/groupstats_pd_protocol_2_contrast.csv")
groupstats_protocol_3_oa = pd.read_csv("../data/groupstats_oa_protocol_3_contrast.csv")
groupstats_protocol_3_pd = pd.read_csv("../data/groupstats_pd_protocol_3_contrast.csv")

# Get NIRS toolbox output of interest
# Note that high/low performing cluster definitions 
# are different in OA and PD; we want contrast between 
# high performing and low performing
protocol_2_oa_st_walk = groupstats_protocol_2_oa[
    groupstats_protocol_2_oa["cond"] == "-Straight_walking:cluster_1+Straight_walking:cluster_2"
].copy()
protocol_2_oa_st_nav = groupstats_protocol_2_oa[
    groupstats_protocol_2_oa["cond"] == "-Navigated_walking:cluster_1+Navigated_walking:cluster_2"
].copy()
protocol_2_pd_st_walk = groupstats_protocol_2_pd[
    groupstats_protocol_2_pd["cond"] == "Straight_walking:cluster_1-Straight_walking:cluster_2"
].copy()
protocol_2_pd_st_nav = groupstats_protocol_3_oa[
    groupstats_protocol_2_pd["cond"] == "Navigated_walking:cluster_1-Navigated_walking:cluster_2"
].copy()

protocol_3_oa_st_nav = groupstats_protocol_3_oa[
    groupstats_protocol_3_oa["cond"] == "-Navigation:cluster_1+Navigation:cluster_2"
].copy()
protocol_3_oa_dt_nav = groupstats_protocol_3_oa[
    groupstats_protocol_3_oa["cond"] == "-Navigation_and_Aud_Stroop:cluster_1+Navigation_and_Aud_Stroop:cluster_2"
].copy()
protocol_3_pd_st_nav = groupstats_protocol_3_pd[
    groupstats_protocol_3_pd["cond"] == "Navigation:cluster_1-Navigation:cluster_2"
].copy()
protocol_3_pd_dt_nav = groupstats_protocol_3_pd[
    groupstats_protocol_3_pd["cond"] == "Navigation_and_Aud_Stroop:cluster_1-Navigation_and_Aud_Stroop:cluster_2"
].copy()



# Label output and project to surface space

In [7]:
# Collect all output of interest
contrast_dict = {
    "p2_oa_st_walk": protocol_2_oa_st_walk,
    "p2_oa_st_nav": protocol_2_oa_st_nav,
    "p2_pd_st_walk": protocol_2_pd_st_walk,
    "p2_pd_st_nav": protocol_2_pd_st_nav,
    
    "p3_oa_st_nav": protocol_3_oa_st_nav,
    "p3_oa_dt_nav": protocol_3_oa_dt_nav,
    "p3_pd_st_nav": protocol_3_pd_st_nav,
    "p3_pd_dt_nav": protocol_3_pd_dt_nav,
    
    "p2_oa_st_walk_cluster_1": protocol_2_oa_st_walk_cluster_1,
    "p2_oa_st_walk_cluster_2": protocol_2_oa_st_walk_cluster_2,
    "p2_oa_st_nav_cluster_1": protocol_2_oa_st_nav_cluster_1,
    "p2_oa_st_nav_cluster_2": protocol_2_oa_st_nav_cluster_2,
    
    "p2_pd_st_walk_cluster_1": protocol_2_pd_st_walk_cluster_1,
    "p2_pd_st_walk_cluster_2": protocol_2_pd_st_walk_cluster_2,
    "p2_pd_st_nav_cluster_1": protocol_2_pd_st_nav_cluster_1,
    "p2_pd_st_nav_cluster_2": protocol_2_pd_st_nav_cluster_2,
    
    "p3_oa_st_nav_cluster_1": protocol_3_oa_st_nav_cluster_1,
    "p3_oa_st_nav_cluster_2": protocol_3_oa_st_nav_cluster_2,
    "p3_oa_dt_nav_cluster_1": protocol_3_oa_dt_nav_cluster_1,
    "p3_oa_dt_nav_cluster_2": protocol_3_oa_dt_nav_cluster_2,
    
    "p3_pd_st_nav_cluster_1": protocol_3_pd_st_nav_cluster_1,
    "p3_pd_st_nav_cluster_2": protocol_3_pd_st_nav_cluster_2,
    "p3_pd_dt_nav_cluster_1": protocol_3_pd_dt_nav_cluster_1,
    "p3_pd_dt_nav_cluster_2": protocol_3_pd_dt_nav_cluster_2,
}

# Adot order
adot_channels = Adot_long.channel.values

# Output containers
img_proj_dict = {}
plotter_dict = {}
surf_dict = {}

for label, t_val_df in contrast_dict.items():

    # Set up channel column to match Adot order
    t_val_df = t_val_df.copy()

    t_val_df["channel"] = (
        "S" + t_val_df["source"].astype(str)
        + "D" + t_val_df["detector"].astype(str)
    )

    # Reorder dataframe to Adot order
    t_val_df_reordered = (
        t_val_df
        .set_index("channel")
        .loc[adot_channels]
        .reset_index()
    )

    # Get t values and encapsulate in xarray
    t_vals = t_val_df_reordered["tstat"].to_numpy()

    t_map = xr.DataArray(
        t_vals,
        dims=["channel"],
        coords={"channel": shared_channels},
    )

    # Project using forward matrix
    img_proj = (Adot_long * t_map).sum("channel")

    img_proj = (
        img_proj
        .transpose("wavelength", "vertex")
        .rename({"wavelength": "chromo"})
        .assign_coords(chromo=["HbO", "HbR"])
    )

    img_proj_dict[label] = img_proj

    # Plot reconstruction
    clim = (-3, 3)

    p0, surf, _ = cedalion.vis.anatomy.image_recon(
        img_proj,
        colin_ijk_labeled,
        view_type="hbo_brain",
        view_position="anterior",
        off_screen=False,
        show_scalar_bar=False,
        clim=clim,
        title_str=label,
    )

    # BA46 boundaries
    roi_labels = ["right_BA46", "left_BA46"]

    boundary_edges = get_parcel_boundary_edges(
        colin_ijk_labeled.brain,
        "parcel_brodmann",
        labels=roi_labels,
    )

    lines = np.hstack(
        [np.full((len(boundary_edges), 1), 2), boundary_edges]
    ).astype(np.int64)

    boundary_poly = pv.PolyData(
        surf.points,
        lines=lines,
    )

    p0.add_mesh(
        boundary_poly,
        color="lime",
        line_width=3,
        render_lines_as_tubes=False,
        pickable=False,
    )

    plotter_dict[label] = p0
    surf_dict[label] = surf

# Plot activation and parcellation

In [8]:
for label, p0 in plotter_dict.items():
    p0.show()

for label, p0 in plotter_dict.items():
    p0.reset_camera()
    p0.camera.zoom(2.0)
    p0.window_size = (1200, 800)
    p0.save_graphic(f"../figures/condition_contrast_figures/{label}.svg")

2026-08-07 14:05:55.219 Python[57709:10060070] ApplePersistenceIgnoreState: Existing state will not be touched. New state will be written to /var/folders/m4/lwl45ymj7ygb_w4t_xhm9d6c0000gp/T/org.python.python.savedState


Widget(value='<iframe src="http://localhost:54538/index.html?ui=P_0x12fb60310_1&reconnect=auto" class="pyvista…

Widget(value='<iframe src="http://localhost:54538/index.html?ui=P_0x12fc32e50_2&reconnect=auto" class="pyvista…

Widget(value='<iframe src="http://localhost:54538/index.html?ui=P_0x12fb77050_3&reconnect=auto" class="pyvista…

Widget(value='<iframe src="http://localhost:54538/index.html?ui=P_0x12ffe9050_4&reconnect=auto" class="pyvista…

...

Widget(value='<iframe src="http://localhost:54538/index.html?ui=P_0x130073510_21&reconnect=auto" class="pyvist…

Widget(value='<iframe src="http://localhost:54538/index.html?ui=P_0x149005750_22&reconnect=auto" class="pyvist…

Widget(value='<iframe src="http://localhost:54538/index.html?ui=P_0x12fd74610_23&reconnect=auto" class="pyvist…

Widget(value='<iframe src="http://localhost:54538/index.html?ui=P_0x1300aac90_24&reconnect=auto" class="pyvist…

 JS Error => Error compiling shader '#version 300 es
#define attribute in
#define textureCube texture
#define texture2D texture
#define textureCubeLod textureLod
#define texture2DLod textureLod


#ifdef GL_FRAGMENT_PRECISION_HIGH
precision highp float;
precision highp int;
#else
precision mediump float;
precision mediump int;
#endif

/*=========================================================================

  Program:   Visualization Toolkit
  Module:    vtkPolyDataVS.glsl

  Copyright (c) Ken Martin, Will Schroeder, Bill Lorensen
  All rights reserved.
  See Copyright.txt or http://www.kitware.com/Copyright.htm for details.

     This software is distributed WITHOUT ANY WARRANTY; without even
     the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR
     PURPOSE.  See the above copyright notice for more information.

=========================================================================*/

attribute vec4 vertexMC;

// frag position in VC
out vec4 vertexVCVSOutput;
